COM OPTUNA

In [1]:
# Instalação e importação de Bibliotecas
# pip install pandas scikit-learn optuna

import pandas as pd
import numpy
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import optuna

c:\Users\ana_v\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


O CÓDIGO ABAIXO FOI RODADO APENAS UMA ÚNICA VEZ, PARA OBTER O CSV
FAVOR NÃO RODAR NOVAMENTE

In [ ]:
# Carregar os dados e renomear as colunas
data = pd.read_csv("C:/Users/ana_v/OneDrive/Documentos/Repositórios/wdbc.csv", header=None)
col_names = ["ID", "Diagnosis", "radius1", "texture1", "perimeter1", "area1",
             "smoothness1", "compactness1", "concavity1", "concave_points1",
             "symmetry1", "fractal_dimension1", "radius2", "texture2",
             "perimeter2", "area2", "smoothness2", "compactness2", "concavity2",
             "concave_points2", "symmetry2", "fractal_dimension2", "radius3",
             "texture3", "perimeter3", "area3", "smoothness3", "compactness3",
             "concavity3", "concave_points3", "symmetry3", "fractal_dimension3"]
data.columns = col_names

# Selecionar dados para o modelo
data_model = data.drop(columns=["ID"])
data_model['Diagnosis'] = data_model['Diagnosis'].map({'M': 1, 'B': 0}) 

# Dividir os dados em treino e teste
train_data, test_data = train_test_split(data_model, test_size=0.2, random_state=123)
X_train = train_data.drop(columns=["Diagnosis"])
y_train = train_data["Diagnosis"]
X_test = test_data.drop(columns=["Diagnosis"])
y_test = test_data["Diagnosis"]

# Salvar os conjuntos de treino e teste em arquivos CSV
X_train.to_csv("X_train.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

In [2]:
X_train = pd.read_csv("C:/Users/ana_v/OneDrive/Área de Trabalho/Doutorado/ERMAC/X_train.csv")
y_train = pd.read_csv("C:/Users/ana_v/OneDrive/Área de Trabalho/Doutorado/ERMAC/y_train.csv").squeeze() 
X_test = pd.read_csv("C:/Users/ana_v/OneDrive/Área de Trabalho/Doutorado/ERMAC/X_test.csv")
y_test = pd.read_csv("C:/Users/ana_v/OneDrive/Área de Trabalho/Doutorado/ERMAC/y_test.csv").squeeze()

In [3]:
# Função objetivo para otimizar o SVM
def objective_svm(trial):
    svc_c = trial.suggest_loguniform('svc_c', 1e-5, 1e2)
    svc_kernel = trial.suggest_categorical('svc_kernel', ['linear', 'poly', 'rbf', 'sigmoid'])
    
    model_svm = SVC(C=svc_c, kernel=svc_kernel, random_state=123)
    score = cross_val_score(model_svm, X_train, y_train, cv=10, scoring='accuracy').mean()
    return score

# Função objetivo para otimizar o Random Forest
def objective_rf(trial):
    rf_n_estimators = trial.suggest_int('rf_n_estimators', 50, 200)
    rf_max_depth = trial.suggest_int('rf_max_depth', 10, 50)
    rf_min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 20)
    
    model_rf = RandomForestClassifier(n_estimators=rf_n_estimators, max_depth=rf_max_depth,
                                      min_samples_split=rf_min_samples_split, random_state=123)
    score = cross_val_score(model_rf, X_train, y_train, cv=10, scoring='accuracy').mean()
    return score

# Função objetivo para otimizar a Regressão Logística
def objective_lr(trial):
    lr_c = trial.suggest_loguniform('lr_c', 1e-5, 1e2)
    lr_solver = trial.suggest_categorical('lr_solver', ['newton-cg', 'lbfgs', 'liblinear', 'saga'])
    
    model_lr = LogisticRegression(C=lr_c, solver=lr_solver, max_iter=10000, random_state=123)
    score = cross_val_score(model_lr, X_train, y_train, cv=10, scoring='accuracy').mean()
    return score

# Estudar otimização com Optuna para SVM
study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=10)
best_params_svm = study_svm.best_params
print("Melhores parâmetros para SVM:", best_params_svm)

# Estudar otimização com Optuna para Random Forest
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=10)
best_params_rf = study_rf.best_params
print("Melhores parâmetros para Random Forest:", best_params_rf)

# Estudar otimização com Optuna para Regressão Logística
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=10)
best_params_lr = study_lr.best_params
print("Melhores parâmetros para Regressão Logística:", best_params_lr)

[I 2025-01-27 11:04:50,043] A new study created in memory with name: no-name-fbecfd56-e2b3-4805-a710-8d24628b3f68
C:\Users\ana_v\AppData\Local\Temp\ipykernel_13788\232050405.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  svc_c = trial.suggest_loguniform('svc_c', 1e-5, 1e2)
[I 2025-01-27 11:04:50,120] Trial 0 finished with value: 0.8965217391304348 and parameters: {'svc_c': 0.3625185252485568, 'svc_kernel': 'rbf'}. Best is trial 0 with value: 0.8965217391304348.
C:\Users\ana_v\AppData\Local\Temp\ipykernel_13788\232050405.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  svc_c = trial.suggest_loguniform('svc_c', 1e-5, 1e2)
[I 2025-01-27 11:04:50,188] Trial 1 finish

Melhores parâmetros para SVM: {'svc_c': 8.374511242127847, 'svc_kernel': 'linear'}


[I 2025-01-27 11:05:21,308] Trial 0 finished with value: 0.9407246376811594 and parameters: {'rf_n_estimators': 60, 'rf_max_depth': 13, 'rf_min_samples_split': 10}. Best is trial 0 with value: 0.9407246376811594.
[I 2025-01-27 11:05:23,568] Trial 1 finished with value: 0.9494685990338164 and parameters: {'rf_n_estimators': 143, 'rf_max_depth': 20, 'rf_min_samples_split': 5}. Best is trial 1 with value: 0.9494685990338164.
[I 2025-01-27 11:05:25,202] Trial 2 finished with value: 0.945072463768116 and parameters: {'rf_n_estimators': 112, 'rf_max_depth': 33, 'rf_min_samples_split': 10}. Best is trial 1 with value: 0.9494685990338164.
[I 2025-01-27 11:05:27,352] Trial 3 finished with value: 0.945072463768116 and parameters: {'rf_n_estimators': 140, 'rf_max_depth': 33, 'rf_min_samples_split': 12}. Best is trial 1 with value: 0.9494685990338164.
[I 2025-01-27 11:05:28,921] Trial 4 finished with value: 0.9516425120772947 and parameters: {'rf_n_estimators': 98, 'rf_max_depth': 22, 'rf_min_samp

Melhores parâmetros para Random Forest: {'rf_n_estimators': 112, 'rf_max_depth': 15, 'rf_min_samples_split': 2}


C:\Users\ana_v\AppData\Local\Temp\ipykernel_13788\232050405.py:23: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr_c = trial.suggest_loguniform('lr_c', 1e-5, 1e2)
[I 2025-01-27 11:05:37,001] Trial 0 finished with value: 0.9339130434782609 and parameters: {'lr_c': 0.0011066557245125671, 'lr_solver': 'newton-cg'}. Best is trial 0 with value: 0.9339130434782609.
C:\Users\ana_v\AppData\Local\Temp\ipykernel_13788\232050405.py:23: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr_c = trial.suggest_loguniform('lr_c', 1e-5, 1e2)
[I 2025-01-27 11:05:37,802] Trial 1 finished with value: 0.9426570048309181 and parameters: {'lr_c': 0.11460781752353713, 'lr_solver': 'newton-cg'}. Be

Melhores parâmetros para Regressão Logística: {'lr_c': 1.0471078398332012, 'lr_solver': 'newton-cg'}


In [4]:
# Treinar e avaliar o modelo SVM com os melhores parâmetros
best_model_svm = SVC(C=best_params_svm['svc_c'], kernel=best_params_svm['svc_kernel'], random_state=123)
best_model_svm.fit(X_train, y_train)
predictions_svm = best_model_svm.predict(X_test)

print("\nResultados do SVM:")
print("Matriz de Confusão:\n", confusion_matrix(y_test, predictions_svm))
print("Acurácia:", accuracy_score(y_test, predictions_svm))
print("Precisão:", precision_score(y_test, predictions_svm))
print("Recall:", recall_score(y_test, predictions_svm))
print("F1 Score:", f1_score(y_test, predictions_svm))

# Treinar e avaliar o modelo Random Forest com os melhores parâmetros
best_model_rf = RandomForestClassifier(
    n_estimators=best_params_rf['rf_n_estimators'],
    max_depth=best_params_rf['rf_max_depth'],
    min_samples_split=best_params_rf['rf_min_samples_split'],
    random_state=123
)
best_model_rf.fit(X_train, y_train)
predictions_rf = best_model_rf.predict(X_test)

print("\nResultados do Random Forest:")
print("Matriz de Confusão:\n", confusion_matrix(y_test, predictions_rf))
print("Acurácia:", accuracy_score(y_test, predictions_rf))
print("Precisão:", precision_score(y_test, predictions_rf))
print("Recall:", recall_score(y_test, predictions_rf))
print("F1 Score:", f1_score(y_test, predictions_rf))

# Treinar e avaliar o modelo Regressão Logística com os melhores parâmetros
best_model_lr = LogisticRegression(
    C=best_params_lr['lr_c'],
    solver=best_params_lr['lr_solver'],
    max_iter=10000,
    random_state=123
)
best_model_lr.fit(X_train, y_train)
predictions_lr = best_model_lr.predict(X_test)

print("\nResultados da Regressão Logística:")
print("Matriz de Confusão:\n", confusion_matrix(y_test, predictions_lr))
print("Acurácia:", accuracy_score(y_test, predictions_lr))
print("Precisão:", precision_score(y_test, predictions_lr))
print("Recall:", recall_score(y_test, predictions_lr))
print("F1 Score:", f1_score(y_test, predictions_lr))


Resultados do SVM:
Matriz de Confusão:
 [[73  0]
 [ 2 39]]
Acurácia: 0.9824561403508771
Precisão: 1.0
Recall: 0.9512195121951219
F1 Score: 0.975

Resultados do Random Forest:
Matriz de Confusão:
 [[73  0]
 [ 1 40]]
Acurácia: 0.9912280701754386
Precisão: 1.0
Recall: 0.975609756097561
F1 Score: 0.9876543209876543

Resultados da Regressão Logística:
Matriz de Confusão:
 [[73  0]
 [ 2 39]]
Acurácia: 0.9824561403508771
Precisão: 1.0
Recall: 0.9512195121951219
F1 Score: 0.975
